# Freemarket Medallion Pipeline — orchestrator

This notebook **orchestrates and narrates**; all transformation logic lives in the tested
`src/` package (see `plan/SPEC.md` § Engineering Standards). Run top-to-bottom
(Restart & Run All) from an empty warehouse.

Layers so far: **Bronze** (`raw` monthly → `live` consolidated + counterparty/fees) →
**Silver `core`** (companies + groups unpick; exchange-rate points landed; FX rate attached).

In [1]:
import sys
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src').is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src import config, warehouse, bronze, silver_core
from src.pipeline import build_foundation
config.WAREHOUSE_PATH

PosixPath('/Users/hallabehery/Documents/My Projects/FM-Data-Engineer-Test-HS/submission/warehouse.duckdb')

## 1. Foundation — six schemas

In [2]:
con = warehouse.connect()
build_foundation(con)

22:37:47 | INFO  | [foundation] schemas ready: raw, live, core, shape, data_mart, curated


('raw', 'live', 'core', 'shape', 'data_mart', 'curated')

## 2. Bronze `raw` — transactional sheets split per month

In [3]:
for report in bronze.build_raw_monthly(con):
    print(report)

22:37:47 | INFO  | [bronze.raw.deposits] in=6383 out=6383 kept=6383 quarantined=0 dropped=0 conserved=yes


22:37:47 | INFO  | [bronze.raw.withdrawals] in=6599 out=6599 kept=6599 quarantined=0 dropped=0 conserved=yes


[bronze.raw.deposits] in=6383 out=6383 kept=6383 quarantined=0 dropped=0 conserved=yes
[bronze.raw.withdrawals] in=6599 out=6599 kept=6599 quarantined=0 dropped=0 conserved=yes


## 3. Bronze `live` — consolidate + land counterparty & fees

In [4]:
for report in bronze.build_live(con):
    print(report)

22:37:47 | INFO  | [bronze.live.deposits] in=6383 out=6383 kept=6383 quarantined=0 dropped=0 conserved=yes


22:37:47 | INFO  | [bronze.live.withdrawals] in=6599 out=6599 kept=6599 quarantined=0 dropped=0 conserved=yes


22:37:47 | INFO  | [bronze.live.counterparty] in=1585 out=1585 kept=1585 quarantined=0 dropped=0 conserved=yes


22:37:47 | INFO  | [bronze.live.fees] in=21921 out=21921 kept=21921 quarantined=0 dropped=0 conserved=yes


[bronze.live.deposits] in=6383 out=6383 kept=6383 quarantined=0 dropped=0 conserved=yes
[bronze.live.withdrawals] in=6599 out=6599 kept=6599 quarantined=0 dropped=0 conserved=yes
[bronze.live.counterparty] in=1585 out=1585 kept=1585 quarantined=0 dropped=0 conserved=yes
[bronze.live.fees] in=21921 out=21921 kept=21921 quarantined=0 dropped=0 conserved=yes


## 4. Silver `core` — companies unpick

In [5]:
print(silver_core.build_companies(con))

22:37:47 | INFO  | [silver.core.companies] in=44 out=44 kept=44 quarantined=0 dropped=0 conserved=yes


[silver.core.companies] in=44 out=44 kept=44 quarantined=0 dropped=0 conserved=yes


## 5. Silver `core` — groups unpick

In [6]:
print(silver_core.build_groups(con))

22:37:47 | INFO  | [silver.core.groups] in=13 out=13 kept=13 quarantined=0 dropped=0 conserved=yes


[silver.core.groups] in=13 out=13 kept=13 quarantined=0 dropped=0 conserved=yes


## 6. Silver `core` — exchange-rate points
Land the FX rate points as `core.exchange_rates` (one row per rate interval), so a
transaction's `fx_rate_id` can be traced back to the exact rate that priced it.

In [7]:
print(silver_core.build_exchange_rates(con))

22:37:48 | INFO  | [silver.core.exchange_rates] in=161342 out=161342 kept=161342 quarantined=0 dropped=0 conserved=yes


[silver.core.exchange_rates] in=161342 out=161342 kept=161342 quarantined=0 dropped=0 conserved=yes


In [8]:
con.sql('''
    SELECT currency, rate_id, valid_from, valid_till, rate
    FROM core.exchange_rates WHERE currency = 'EUR'
    ORDER BY valid_from LIMIT 5
''').df()

,currency,rate_id,valid_from,valid_till,rate
0,EUR,13757179,2025-07-01 00:59:00.720000+01:00,2025-07-01 01:00:02.423000+01:00,0.858368
1,EUR,13757189,2025-07-01 01:00:02.423000+01:00,2025-07-01 01:01:00.740000+01:00,0.858389
2,EUR,13757230,2025-07-01 01:01:00.740000+01:00,2025-07-01 01:02:00.823000+01:00,0.858354
3,EUR,13757236,2025-07-01 01:02:00.823000+01:00,2025-07-01 01:03:00.723000+01:00,0.858368
4,EUR,13757240,2025-07-01 01:03:00.723000+01:00,2025-07-01 01:04:00.780000+01:00,0.858299


## 7. Silver `core` — FX rate attached to transactions & fees
Attach the as-of rate to every deposit, withdrawal and fee via the tested FX unit
(`src/fx.py`). Settlement instant = `Tx Date + Tx Time` (transactions) / fee `Date`
(fees). Rates are **attached, not applied** (GBP comes in Silver `shape`). Each row keeps
`fx_rate`, `fx_rate_id` (lineage) and `fx_quarantine_reason` (kept, not dropped).

In [9]:
for report in silver_core.build_fx_attached(con):
    print(report)

22:37:49 | INFO  | [silver.core.deposits_fx] in=6383 out=6383 kept=6383 quarantined=0 dropped=0 conserved=yes


22:37:49 | INFO  | [silver.core.withdrawals_fx] in=6599 out=6599 kept=6599 quarantined=0 dropped=0 conserved=yes


22:37:49 | INFO  | [silver.core.fees_fx] in=21921 out=21921 kept=21921 quarantined=0 dropped=0 conserved=yes


22:37:49 | INFO  | [silver.core.fees_fx] flagged for FX quarantine (kept, not dropped): 42


[silver.core.deposits_fx] in=6383 out=6383 kept=6383 quarantined=0 dropped=0 conserved=yes
[silver.core.withdrawals_fx] in=6599 out=6599 kept=6599 quarantined=0 dropped=0 conserved=yes
[silver.core.fees_fx] in=21921 out=21921 kept=21921 quarantined=0 dropped=0 conserved=yes


### Lineage — trace a transaction back to the exact rate point used

In [10]:
con.sql('''
    SELECT d."Transaction ID", d."Tx Currency", d."Tx Value (CCY)",
           d.fx_rate_id, d.fx_rate, r.valid_from, r.valid_till
    FROM core.deposits d
    LEFT JOIN core.exchange_rates r ON d.fx_rate_id = r.rate_id
    WHERE d."Tx Currency" <> 'GBP'
    ORDER BY d."Transaction ID" LIMIT 8
''').df()

,Transaction ID,Tx Currency,Tx Value (CCY),fx_rate_id,fx_rate,valid_from,valid_till
0,334619.0,USD,1.600000e+06,13760520,0.726732,2025-07-01 09:33:00.850000+01:00,2025-07-01 09:34:05.500000+01:00
1,334687.0,USD,1.500000e+06,13758265,0.727605,2025-07-01 05:13:12.733000+01:00,2025-07-01 05:47:02.040000+01:00
2,334706.0,USD,2.673270e+04,13758924,0.727875,2025-07-01 07:40:00.870000+01:00,2025-07-01 07:41:04.510000+01:00
3,334719.0,EUR,1.783262e+05,13759097,0.856944,2025-07-01 08:04:03.296000+01:00,2025-07-01 08:09:50.620000+01:00
4,334740.0,EUR,1.500000e+04,13762739,0.858816,2025-07-01 16:12:00.913000+01:00,2025-07-01 16:14:01.563000+01:00
5,334771.0,EUR,5.396000e+03,13760690,0.857017,2025-07-01 10:00:07.273000+01:00,2025-07-01 10:01:00.983000+01:00
6,334776.0,USD,2.000000e+05,13761437,0.725849,2025-07-01 12:05:00.746000+01:00,2025-07-01 12:36:58.273000+01:00
7,334785.0,SEK,3.790666e+06,13760605,0.076937,2025-07-01 09:46:48.923000+01:00,2025-07-01 10:40:16.976000+01:00


### Data-quality — rows flagged for FX quarantine (kept, not dropped), by reason

In [11]:
con.sql('''
    SELECT 'deposits' AS tbl, fx_quarantine_reason, COUNT(*) n FROM core.deposits WHERE fx_quarantine_reason IS NOT NULL GROUP BY 1,2
    UNION ALL SELECT 'withdrawals', fx_quarantine_reason, COUNT(*) FROM core.withdrawals WHERE fx_quarantine_reason IS NOT NULL GROUP BY 1,2
    UNION ALL SELECT 'fees', fx_quarantine_reason, COUNT(*) FROM core.fees WHERE fx_quarantine_reason IS NOT NULL GROUP BY 1,2
    ORDER BY 1,2
''').df()

,tbl,fx_quarantine_reason,n
0,fees,fx_missing_currency,42


## Inspect the built warehouse

In [12]:
con.close()
import duckdb
inspect = duckdb.connect(str(config.WAREHOUSE_PATH), read_only=True)
tables = inspect.sql('''
    SELECT table_schema, COUNT(*) AS n_tables
    FROM information_schema.tables GROUP BY table_schema ORDER BY table_schema
''').df()
inspect.close()
tables

,table_schema,n_tables
0,core,6
1,live,4
2,raw,12
